# 06 — Week 2 Projections

This notebook converts the updated weekly team-strength ratings into game level projections for the selected target week.

The original preseason notebooks and preseason projections remain frozen.

For each target week matchup, this notebook:

- loads the updated weekly team strengths from Notebook 04
- preserves the original preseason Week 2 projection as a baseline
- recalculates expected margin using weekly strength, historical home field advantage, and rest
- converts expected margin into win probability using historical NFL margin variability
- carries forward the frozen preseason scoring baseline created for Week 1
- adds the new in season offensive and defensive scoring evidence
- produces projected scores, model spreads, ATS comparisons, totals comparisons, and final output tables
- saves all outputs only inside the weekly layer

The market is used only for comparison. It never enters the model projection itself.


In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from scipy.stats import norm
from sklearn.linear_model import LinearRegression


## Paths and Weekly Settings

For Week 2, `TARGET_WEEK = 2`.

Future weeks use the same notebook by changing the target week after the preceding weekly notebooks have been rerun.


In [12]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"
WEEKLY_OUTPUT_DIR = PROJECT_ROOT / "weekly_projections" / "outputs"

SEASON = 2026
TARGET_WEEK = 5

WEEK_OUTPUT_DIR = (
    WEEKLY_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}"
)

WEEK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEEKLY_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)

INJURY_ADJUSTED_STRENGTH_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

TARGET_SCHEDULE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_schedule.parquet"
)

PRESEASON_BASELINE_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_preseason_baseline.parquet"
)

WEEK1_PROJECTIONS_PATH = (
    WEEKLY_DATA_DIR
    / "week_01_projections.parquet"
)

# Load Weekly Inputs

Notebook 04 supplies the updated ratings. Notebook 02 supplies the target week schedule and frozen preseason comparison.


In [13]:
weekly_strength = pd.read_parquet(WEEKLY_STRENGTH_PATH)

injury_adjusted_strength = pd.read_parquet(
    INJURY_ADJUSTED_STRENGTH_PATH
)

target_schedule = pd.read_parquet(
    TARGET_SCHEDULE_PATH
)

preseason_baseline = pd.read_parquet(
    PRESEASON_BASELINE_PATH
)

target_schedule = target_schedule[
    target_schedule["home_score"].isna()
    & target_schedule["away_score"].isna()
].copy()


preseason_baseline = preseason_baseline[
    preseason_baseline["game_id"].isin(
        target_schedule["game_id"]
    )
].copy()


print(
    "Sequential weekly strengths:",
    weekly_strength.shape
)

print(
    "Injury-adjusted strengths:",
    injury_adjusted_strength.shape
)

print(
    "Remaining target-week games:",
    target_schedule.shape
)

print(
    "Frozen preseason games:",
    preseason_baseline.shape
)

assert len(weekly_strength) == 32
assert weekly_strength["team"].nunique() == 32

assert len(injury_adjusted_strength) == 32
assert injury_adjusted_strength["team"].nunique() == 32

assert len(target_schedule) == len(
    preseason_baseline
)

required_injury_columns = [
    "team",
    "weekly_team_strength",
    "injury_adjustment",
    "injury_adjusted_team_strength"
]

missing_injury_columns = [
    col
    for col in required_injury_columns
    if col not in injury_adjusted_strength.columns
]

assert not missing_injury_columns, (
    "Missing injury-adjusted columns: "
    + ", ".join(missing_injury_columns)
)


print(
    "Using sequential ratings as the pre-injury baseline "
    "and injury-adjusted ratings for Week "
    f"{TARGET_WEEK} projections."
)

Sequential weekly strengths: (32, 25)
Injury-adjusted strengths: (32, 30)
Remaining target-week games: (15, 46)
Frozen preseason games: (15, 16)
Using sequential ratings as the pre-injury baseline and injury-adjusted ratings for Week 5 projections.


### Injury adjusted game strength

For Week 2, the `home_weekly_team_strength` and `away_weekly_team_strength` values now include the conservative injury adjustment from Notebook 05.

The injury adjustment remains visible separately in the saved team strength file, so we can distinguish:

**preseason → performance update → injury update → final Week 2 rating**


# Attach Updated Team Strength

The matchup calculation is identical in structure to the original preseason game prediction model. The only change is the source of team strength:

`preseason team_strength` → `weekly_team_strength`

This isolates the in season update without changing the original model.


In [14]:
strength_lookup = injury_adjusted_strength[
    [
        "team",
        "weekly_team_strength",
        "injury_adjustment",
        "injury_adjusted_team_strength",
        "preseason_team_strength",
        "team_strength_change",
        "cumulative_offense_adjustment",
        "cumulative_defense_adjustment"
    ]
].copy()


games = target_schedule.copy()

home_lookup = strength_lookup.rename(
    columns={
        "team": "home_team",

        "weekly_team_strength":
            "home_pre_injury_team_strength",

        "injury_adjustment":
            "home_injury_adjustment",

        "injury_adjusted_team_strength":
            "home_weekly_team_strength",

        "preseason_team_strength":
            "home_preseason_team_strength",

        "team_strength_change":
            "home_strength_change",

        "cumulative_offense_adjustment":
            "home_offense_adjustment",

        "cumulative_defense_adjustment":
            "home_defense_adjustment"
    }
)

away_lookup = strength_lookup.rename(
    columns={
        "team": "away_team",

        "weekly_team_strength":
            "away_pre_injury_team_strength",

        "injury_adjustment":
            "away_injury_adjustment",

        "injury_adjusted_team_strength":
            "away_weekly_team_strength",

        "preseason_team_strength":
            "away_preseason_team_strength",

        "team_strength_change":
            "away_strength_change",

        "cumulative_offense_adjustment":
            "away_offense_adjustment",

        "cumulative_defense_adjustment":
            "away_defense_adjustment"
    }
)


games = (
    games
    .merge(
        home_lookup,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_lookup,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

games["weekly_neutral_strength_diff"] = (
    games["home_weekly_team_strength"]
    - games["away_weekly_team_strength"]
)

games["pre_injury_neutral_strength_diff"] = (
    games["home_pre_injury_team_strength"]
    - games["away_pre_injury_team_strength"]
)


games["preseason_neutral_strength_diff"] = (
    games["home_preseason_team_strength"]
    - games["away_preseason_team_strength"]
)

assert games[
    "home_weekly_team_strength"
].notna().all()

assert games[
    "away_weekly_team_strength"
].notna().all()

assert games[
    "home_injury_adjustment"
].notna().all()

assert games[
    "away_injury_adjustment"
].notna().all()

# Home Field and Rest

These effects are recalculated using the same historical 2015–2025 methodology as the frozen preseason game prediction notebook.

They are not retuned using Week 1.


In [15]:
schedule_clean = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

historical_home = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & (pl.col("location") == "Home")
        & pl.col("result").is_not_null()
    )
)

HOME_FIELD_ADVANTAGE = (
    historical_home
    .select(pl.col("result").mean())
    .item()
)

historical_rest = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
        & pl.col("home_rest").is_not_null()
        & pl.col("away_rest").is_not_null()
    )
    .select(["result", "home_rest", "away_rest", "location"])
    .to_pandas()
)

historical_rest["rest_diff"] = (
    historical_rest["home_rest"]
    - historical_rest["away_rest"]
).clip(-7, 7)

historical_rest["home_field"] = (
    historical_rest["location"] == "Home"
).astype(int)

rest_model = LinearRegression()
rest_model.fit(
    historical_rest[["home_field", "rest_diff"]],
    historical_rest["result"]
)

REST_POINT_VALUE = rest_model.coef_[1]

print("Historical HFA:", round(HOME_FIELD_ADVANTAGE, 3))
print("Rest points/day:", round(REST_POINT_VALUE, 3))


Historical HFA: 1.764
Rest points/day: 0.181


In [16]:
games["home_field_adjustment"] = np.where(
    games["location"] == "Home",
    HOME_FIELD_ADVANTAGE,
    0.0
)

games["rest_diff"] = (
    games["home_rest"]
    - games["away_rest"]
).clip(-7, 7)

games["rest_adjustment"] = (
    games["rest_diff"]
    * REST_POINT_VALUE
)

games["expected_home_margin"] = (
    games["weekly_neutral_strength_diff"]
    + games["home_field_adjustment"]
    + games["rest_adjustment"]
)


# Win Probabilities

Historical 2015–2025 game-margin variability is retained exactly as the uncertainty scale. Week 1 is not used to recalibrate probability confidence.


In [17]:
historical_margin_std = (
    schedule_clean
    .filter(
        (pl.col("season") >= 2015)
        & (pl.col("season") <= 2025)
        & pl.col("result").is_not_null()
    )
    .select(pl.col("result").std())
    .item()
)

games["home_win_probability"] = norm.cdf(
    games["expected_home_margin"]
    / historical_margin_std
)

games["away_win_probability"] = (
    1 - games["home_win_probability"]
)

games["predicted_winner"] = np.where(
    games["home_win_probability"] >= 0.50,
    games["home_team"],
    games["away_team"]
)

games["predicted_win_probability"] = np.maximum(
    games["home_win_probability"],
    games["away_win_probability"]
)

print("Historical margin SD:", round(historical_margin_std, 3))


Historical margin SD: 14.201


# Preseason vs Weekly Margin Movement

The frozen preseason Week 2 projection is attached for diagnostics only.

This makes it possible to see exactly how much the Week 1 evidence changed each Week 2 matchup.


In [18]:
preseason_compare = preseason_baseline[
    [
        "game_id",
        "expected_home_margin",
        "predicted_winner",
        "predicted_win_probability"
    ]
].rename(columns={
    "expected_home_margin": "preseason_expected_home_margin",
    "predicted_winner": "preseason_predicted_winner",
    "predicted_win_probability": "preseason_predicted_win_probability"
})

games = games.merge(
    preseason_compare,
    on="game_id",
    how="left",
    validate="one_to_one"
)

games["margin_change_from_preseason"] = (
    games["expected_home_margin"]
    - games["preseason_expected_home_margin"]
)

display(
    games[
        [
            "away_team",
            "home_team",
            "preseason_expected_home_margin",
            "expected_home_margin",
            "margin_change_from_preseason",
            "preseason_predicted_winner",
            "predicted_winner",
            "predicted_win_probability"
        ]
    ].round(3)
)


,away_team,home_team,preseason_expected_home_margin,expected_home_margin,margin_change_from_preseason,preseason_predicted_winner,predicted_winner,predicted_win_probability
0,TB,DAL,0.776,3.218,2.442,DAL,DAL,0.590
1,BAL,ATL,-2.823,-1.016,1.807,BAL,BAL,0.529
2,CHI,GB,4.352,-0.569,-4.921,GB,CHI,0.516
3,CIN,MIA,0.060,-4.031,-4.091,MIA,CIN,0.612
4,CLE,NYJ,-0.563,-0.188,0.375,CLE,CLE,0.505
5,DEN,LAC,-0.895,-2.491,-1.595,DEN,DEN,0.570
6,DET,ARI,-5.933,-3.009,2.924,DET,DET,0.584
7,HOU,TEN,-7.794,-4.489,3.305,HOU,HOU,0.624
8,IND,PIT,2.318,0.770,-1.548,PIT,PIT,0.522
9,LV,NE,9.367,5.477,-3.890,NE,NE,0.650


# Frozen Preseason Scoring Ratings

Week 1's saved projection artifact contains the preseason offensive/defensive scoring ratings and personnel scoring adjustments for all 32 teams.

Those values were created before the season and are reused here as the frozen scoring prior. We do not rebuild or modify the preseason notebooks.

The new weekly offense/defense signals are added separately afterward.


In [19]:
week1_frozen = pd.read_parquet(
    WEEK1_PROJECTIONS_PATH
)

home_scoring = week1_frozen[
    [
        "home_team",
        "home_projected_off_ppg",
        "home_projected_def_ppg",
        "home_off_personnel_z",
        "home_def_personnel_z"
    ]
].rename(columns={
    "home_team": "team",
    "home_projected_off_ppg": "preseason_projected_off_ppg",
    "home_projected_def_ppg": "preseason_projected_def_ppg",
    "home_off_personnel_z": "off_personnel_z",
    "home_def_personnel_z": "def_personnel_z"
})

away_scoring = week1_frozen[
    [
        "away_team",
        "away_projected_off_ppg",
        "away_projected_def_ppg",
        "away_off_personnel_z",
        "away_def_personnel_z"
    ]
].rename(columns={
    "away_team": "team",
    "away_projected_off_ppg": "preseason_projected_off_ppg",
    "away_projected_def_ppg": "preseason_projected_def_ppg",
    "away_off_personnel_z": "off_personnel_z",
    "away_def_personnel_z": "def_personnel_z"
})

scoring_prior = (
    pd.concat([home_scoring, away_scoring], ignore_index=True)
    .groupby("team", as_index=False)
    .agg(
        preseason_projected_off_ppg=("preseason_projected_off_ppg", "mean"),
        preseason_projected_def_ppg=("preseason_projected_def_ppg", "mean"),
        off_personnel_z=("off_personnel_z", "mean"),
        def_personnel_z=("def_personnel_z", "mean")
    )
)

assert len(scoring_prior) == 32
display(scoring_prior.head())


,team,preseason_projected_off_ppg,preseason_projected_def_ppg,off_personnel_z,def_personnel_z
0,ARI,22.191473,24.425844,-0.763765,-1.345326
1,ATL,22.032425,23.261103,0.507936,0.513715
2,BAL,25.007820,22.580763,0.442967,0.969729
3,BUF,25.942602,22.289034,1.060099,0.231615
4,CAR,20.533474,23.731353,-1.506345,-0.346889


In [20]:
completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

completed_games = pd.read_parquet(
    completed_games_path
).copy()

completed_games = completed_games[
    completed_games["week"] < TARGET_WEEK
].copy()

home_scoring_actual = completed_games[
    [
        "home_team",
        "home_score",
        "away_score"
    ]
].rename(
    columns={
        "home_team": "team",
        "home_score": "points_for",
        "away_score": "points_against"
    }
)

away_scoring_actual = completed_games[
    [
        "away_team",
        "away_score",
        "home_score"
    ]
].rename(
    columns={
        "away_team": "team",
        "away_score": "points_for",
        "home_score": "points_against"
    }
)

team_scoring_actual = pd.concat(
    [
        home_scoring_actual,
        away_scoring_actual
    ],
    ignore_index=True
)

inseason_scoring = (
    team_scoring_actual
    .groupby("team", as_index=False)
    .agg(
        games_played=("points_for", "size"),
        actual_off_ppg=("points_for", "mean"),
        actual_def_ppg=("points_against", "mean")
    )
)

assert len(inseason_scoring) == 32

assert (
    inseason_scoring["games_played"] == TARGET_WEEK - 1
).all()

display(
    inseason_scoring
    .sort_values("actual_off_ppg", ascending=False)
    .round(2)
)

,team,games_played,actual_off_ppg,actual_def_ppg
3,BUF,4,31.75,26.75
8,DAL,4,30.50,28.00
28,SF,4,30.50,16.00
4,CAR,4,30.25,27.25
10,DET,4,29.75,31.75
15,KC,4,29.50,19.25
2,BAL,4,29.00,24.00
18,LV,4,28.75,21.00
5,CHI,4,28.00,16.25
27,SEA,4,26.25,18.25


In [21]:
scoring_expectations = scoring_prior[
    [
        "team",
        "preseason_projected_off_ppg",
        "preseason_projected_def_ppg"
    ]
].copy()

home_games = completed_games[
    [
        "week",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_games = home_games.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_for",
        "away_score": "points_against"
    }
)

away_games = completed_games[
    [
        "week",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_games = away_games.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_for",
        "home_score": "points_against"
    }
)

team_games = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

opponent_defense = scoring_expectations[
    [
        "team",
        "preseason_projected_def_ppg"
    ]
].rename(
    columns={
        "team": "opponent",
        "preseason_projected_def_ppg":
            "opponent_expected_def_ppg"
    }
)

opponent_offense = scoring_expectations[
    [
        "team",
        "preseason_projected_off_ppg"
    ]
].rename(
    columns={
        "team": "opponent",
        "preseason_projected_off_ppg":
            "opponent_expected_off_ppg"
    }
)

team_games = (
    team_games
    .merge(
        opponent_defense,
        on="opponent",
        how="left",
        validate="many_to_one"
    )
    .merge(
        opponent_offense,
        on="opponent",
        how="left",
        validate="many_to_one"
    )
)

league_preseason_off_ppg = (
    scoring_prior[
        "preseason_projected_off_ppg"
    ].mean()
)

league_preseason_def_ppg = (
    scoring_prior[
        "preseason_projected_def_ppg"
    ].mean()
)

team_games["adjusted_points_for"] = (
    team_games["points_for"]
    + (
        league_preseason_def_ppg
        - team_games["opponent_expected_def_ppg"]
    )
)

team_games["adjusted_points_against"] = (
    team_games["points_against"]
    + (
        league_preseason_off_ppg
        - team_games["opponent_expected_off_ppg"]
    )
)

inseason_scoring_adjusted = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games_played=("points_for", "size"),
        actual_off_ppg=("points_for", "mean"),
        actual_def_ppg=("points_against", "mean"),
        adjusted_off_ppg=(
            "adjusted_points_for",
            "mean"
        ),
        adjusted_def_ppg=(
            "adjusted_points_against",
            "mean"
        )
    )
)

assert len(inseason_scoring_adjusted) == 32

display(
    inseason_scoring_adjusted[
        [
            "team",
            "actual_off_ppg",
            "adjusted_off_ppg",
            "actual_def_ppg",
            "adjusted_def_ppg"
        ]
    ]
    .sort_values(
        "adjusted_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,actual_off_ppg,adjusted_off_ppg,actual_def_ppg,adjusted_def_ppg
3,BUF,31.75,32.54,26.75,25.89
28,SF,30.50,30.52,16.00,15.67
8,DAL,30.50,30.40,28.00,27.86
4,CAR,30.25,30.07,27.25,27.30
15,KC,29.50,29.43,19.25,19.82
10,DET,29.75,29.39,31.75,32.67
18,LV,28.75,29.33,21.00,21.83
2,BAL,29.00,28.07,24.00,24.36
5,CHI,28.00,27.99,16.25,17.48
14,JAX,26.00,26.12,13.25,13.52


In [22]:
scoring_model = scoring_prior.merge(
    inseason_scoring_adjusted,
    on="team",
    how="left",
    validate="one_to_one"
)

PRESEASON_EQUIVALENT_GAMES = 5.0

scoring_model["preseason_weight"] = (
    PRESEASON_EQUIVALENT_GAMES
    / (
        PRESEASON_EQUIVALENT_GAMES
        + scoring_model["games_played"]
    )
)

scoring_model["inseason_weight"] = (
    1.0
    - scoring_model["preseason_weight"]
)

scoring_model["current_off_ppg"] = (
    scoring_model["preseason_weight"]
    * scoring_model["preseason_projected_off_ppg"]
    +
    scoring_model["inseason_weight"]
    * scoring_model["adjusted_off_ppg"]
)

scoring_model["current_def_ppg"] = (
    scoring_model["preseason_weight"]
    * scoring_model["preseason_projected_def_ppg"]
    +
    scoring_model["inseason_weight"]
    * scoring_model["adjusted_def_ppg"]
)

assert len(scoring_model) == 32

assert scoring_model[
    [
        "current_off_ppg",
        "current_def_ppg"
    ]
].notna().all().all()

display(
    scoring_model[
        [
            "team",
            "preseason_projected_off_ppg",
            "adjusted_off_ppg",
            "current_off_ppg",
            "preseason_projected_def_ppg",
            "adjusted_def_ppg",
            "current_def_ppg",
            "preseason_weight",
            "inseason_weight"
        ]
    ]
    .sort_values(
        "current_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,preseason_projected_off_ppg,adjusted_off_ppg,current_off_ppg,preseason_projected_def_ppg,adjusted_def_ppg,current_def_ppg,preseason_weight,inseason_weight
3,BUF,25.94,32.54,28.87,22.29,25.89,23.89,0.56,0.44
10,DET,26.28,29.39,27.66,23.05,32.67,27.32,0.56,0.44
8,DAL,24.71,30.40,27.24,24.75,27.86,26.13,0.56,0.44
28,SF,24.31,30.52,27.07,22.70,15.67,19.57,0.56,0.44
2,BAL,25.01,28.07,26.37,22.58,24.36,23.37,0.56,0.44
15,KC,22.39,29.43,25.52,21.52,19.82,20.77,0.56,0.44
5,CHI,23.20,27.99,25.33,23.18,17.48,20.65,0.56,0.44
27,SEA,24.46,26.02,25.16,21.63,18.47,20.23,0.56,0.44
13,IND,24.31,25.98,25.05,23.56,25.62,24.48,0.56,0.44
14,JAX,23.95,26.12,24.91,22.47,13.52,18.49,0.56,0.44


# Weekly Scoring Update

The in-season offensive and defensive adjustments from Notebook 04 are already shrunk for sample size.

They are added to the frozen scoring prior:

- positive offensive adjustment raises expected scoring
- positive defensive adjustment represents points saved, so it lowers projected points allowed

This keeps the Week 1 scoring evidence separate and interpretable.


In [23]:
scoring_update = scoring_model[
    [
        "team",
        "current_off_ppg",
        "current_def_ppg"
    ]
].copy()

assert len(scoring_update) == 32
assert scoring_update["team"].nunique() == 32

display(
    scoring_update
    .sort_values(
        "current_off_ppg",
        ascending=False
    )
    .round(2)
)

,team,current_off_ppg,current_def_ppg
3,BUF,28.87,23.89
10,DET,27.66,27.32
8,DAL,27.24,26.13
28,SF,27.07,19.57
2,BAL,26.37,23.37
15,KC,25.52,20.77
5,CHI,25.33,20.65
27,SEA,25.16,20.23
13,IND,25.05,24.48
14,JAX,24.91,18.49


# Weekly Projected Total

The matchup total uses the same basic preseason scoring concept: each offense is paired with the opposing defense.

Because the weekly offense/defense inputs are already heavily shrunk, no additional Week 1 multiplier is applied here.

The final projected scores are forced to satisfy both:

- projected score difference = weekly expected margin
- projected score sum = weekly projected total


In [24]:
home_scoring_update = scoring_update.rename(
    columns={
        "team": "home_team",
        "current_off_ppg":
            "home_current_off_ppg",
        "current_def_ppg":
            "home_current_def_ppg"
    }
)

away_scoring_update = scoring_update.rename(
    columns={
        "team": "away_team",
        "current_off_ppg":
            "away_current_off_ppg",
        "current_def_ppg":
            "away_current_def_ppg"
    }
)

In [25]:
games = (
    games
    .merge(
        home_scoring_update,
        on="home_team",
        how="left",
        validate="many_to_one"
    )
    .merge(
        away_scoring_update,
        on="away_team",
        how="left",
        validate="many_to_one"
    )
)

required_scoring_columns = [
    "home_current_off_ppg",
    "home_current_def_ppg",
    "away_current_off_ppg",
    "away_current_def_ppg"
]

assert games[
    required_scoring_columns
].notna().all().all()

games["raw_home_score"] = (
    games["home_current_off_ppg"]
    + games["away_current_def_ppg"]
) / 2

games["raw_away_score"] = (
    games["away_current_off_ppg"]
    + games["home_current_def_ppg"]
) / 2

games["projected_total"] = (
    games["raw_home_score"]
    + games["raw_away_score"]
)

games["projected_home_score"] = (
    games["projected_total"]
    + games["expected_home_margin"]
) / 2

games["projected_away_score"] = (
    games["projected_total"]
    - games["expected_home_margin"]
) / 2


assert games["projected_total"].notna().all()

assert np.allclose(
    games["projected_home_score"]
    - games["projected_away_score"],
    games["expected_home_margin"]
)

assert np.allclose(
    games["projected_home_score"]
    + games["projected_away_score"],
    games["projected_total"]
)


display(
    games[
        [
            "away_team",
            "home_team",
            "away_current_off_ppg",
            "home_current_def_ppg",
            "home_current_off_ppg",
            "away_current_def_ppg",
            "projected_away_score",
            "projected_home_score",
            "projected_total",
            "expected_home_margin"
        ]
    ].round(2)
)

,away_team,home_team,away_current_off_ppg,home_current_def_ppg,home_current_off_ppg,away_current_def_ppg,projected_away_score,projected_home_score,projected_total,expected_home_margin
0,TB,DAL,21.51,26.13,27.24,23.68,23.03,26.25,49.28,3.22
1,BAL,ATL,26.37,23.58,22.97,23.37,24.58,23.56,48.14,-1.02
2,CHI,GB,25.33,24.38,21.09,20.65,23.15,22.58,45.73,-0.57
3,CIN,MIA,24.33,24.54,17.78,22.92,24.41,20.38,44.79,-4.03
4,CLE,NYJ,20.14,22.76,19.69,23.46,21.61,21.42,43.02,-0.19
5,DEN,LAC,21.49,23.90,19.93,22.00,23.07,20.58,43.66,-2.49
6,DET,ARI,27.66,26.47,22.19,27.32,27.42,24.41,51.82,-3.01
7,HOU,TEN,21.78,22.95,17.20,23.28,23.55,19.06,42.61,-4.49
8,IND,PIT,25.05,22.43,21.02,24.48,22.86,23.63,46.49,0.77
9,LV,NE,23.85,20.28,20.52,22.81,19.13,24.60,43.73,5.48


# Market Comparison

nflverse uses positive `spread_line` for a home favorite and negative for an away favorite. The values below are converted to standard sportsbook notation.

Market information is used only after the model projections have been completed.


In [26]:
games["home_spread"] = -games["spread_line"]
games["away_spread"] = games["spread_line"]

games["model_home_spread"] = -games["expected_home_margin"]
games["model_away_spread"] = games["expected_home_margin"]

games["home_ats_edge"] = (
    games["home_spread"]
    - games["model_home_spread"]
)

games["away_ats_edge"] = (
    games["away_spread"]
    - games["model_away_spread"]
)

games["ats_pick"] = np.where(
    games["home_ats_edge"] > games["away_ats_edge"],
    games["home_team"],
    np.where(
        games["away_ats_edge"] > games["home_ats_edge"],
        games["away_team"],
        "PUSH"
    )
)

games["ats_difference"] = (
    games[["home_ats_edge", "away_ats_edge"]]
    .max(axis=1)
)

games["total_difference_signed"] = (
    games["projected_total"]
    - games["total_line"]
)

games["total_difference"] = (
    games["total_difference_signed"].abs()
)

games["total_pick"] = np.where(
    games["total_difference_signed"] > 0,
    "OVER",
    np.where(
        games["total_difference_signed"] < 0,
        "UNDER",
        "PUSH"
    )
)


## Unit Sizing

Unit sizes are assigned from the model's edge over the market.

ATS and totals use separate thresholds because the totals model is newer and requires a larger edge before qualifying as a play.

These units do not change any projections or picks. They provide a consistent way to track performance by model confidence.

In [27]:
def assign_ats_units(edge):
    if pd.isna(edge) or edge < 1.0:
        return 0.0
    elif edge < 2.0:
        return 0.5
    elif edge < 3.0:
        return 1.0
    elif edge < 4.0:
        return 1.5
    else:
        return 2.0


def assign_total_units(edge):
    if pd.isna(edge) or edge < 2.0:
        return 0.0
    elif edge < 3.5:
        return 0.5
    elif edge < 5.0:
        return 1.0
    elif edge < 7.0:
        return 1.5
    else:
        return 2.0


games["ats_units"] = games["ats_difference"].apply(assign_ats_units)

games["total_units"] = games["total_difference"].apply(assign_total_units)


games[
    [
        "away_team",
        "home_team",
        "ats_pick",
        "ats_difference",
        "ats_units",
        "total_pick",
        "total_difference",
        "total_units"
    ]
]

,away_team,home_team,ats_pick,ats_difference,ats_units,total_pick,total_difference,total_units
0,TB,DAL,TB,6.282082,2.0,OVER,1.782369,0.0
1,BAL,ATL,BAL,3.516076,1.5,OVER,3.644288,1.0
2,CHI,GB,GB,2.430920,1.0,OVER,0.225893,0.0
3,CIN,MIA,MIA,2.969176,1.0,OVER,2.287334,0.5
4,CLE,NYJ,CLE,2.688018,1.0,OVER,3.524420,1.0
5,DEN,LAC,LAC,1.009307,0.5,OVER,1.158095,0.0
6,DET,ARI,ARI,1.490560,0.5,UNDER,2.678100,0.5
7,HOU,TEN,TEN,2.511189,1.0,OVER,3.105195,0.5
8,IND,PIT,IND,1.729751,0.5,OVER,1.993250,0.0
9,LV,NE,NE,1.976867,0.5,UNDER,1.771708,0.0


# Display Helpers


In [28]:
def format_spread(team, line):
    if pd.isna(line):
        return "N/A"
    if line > 0:
        return f"{team} +{line:.1f}"
    return f"{team} {line:.1f}"

def model_spread_display(row):
    if row["model_home_spread"] < 0:
        return format_spread(row["home_team"], row["model_home_spread"])
    if row["model_away_spread"] < 0:
        return format_spread(row["away_team"], row["model_away_spread"])
    return "PICK"

def market_spread_display(row):
    if pd.isna(row["spread_line"]):
        return "N/A"
    if row["home_spread"] < 0:
        return format_spread(row["home_team"], row["home_spread"])
    if row["away_spread"] < 0:
        return format_spread(row["away_team"], row["away_spread"])
    return "PICK"

def ats_pick_display(row):
    if row["ats_pick"] == "PUSH":
        return "PUSH"
    if row["ats_pick"] == row["home_team"]:
        return format_spread(row["home_team"], row["home_spread"])
    return format_spread(row["away_team"], row["away_spread"])

games["projected_score"] = (
    games["away_team"]
    + " "
    + games["projected_away_score"].round(1).astype(str)
    + " - "
    + games["home_team"]
    + " "
    + games["projected_home_score"].round(1).astype(str)
)

games["model_spread_display"] = games.apply(
    model_spread_display, axis=1
)

games["market_spread_display"] = games.apply(
    market_spread_display, axis=1
)

games["ats_pick_display"] = games.apply(
    ats_pick_display, axis=1
)


# Final Week Projection Board


In [29]:
projection_board = pd.DataFrame({
    "Matchup": games["away_team"] + " @ " + games["home_team"],
    "Projected Score": games["projected_score"],
    "SU Pick": games["predicted_winner"],
    "Win Prob": games["predicted_win_probability"],
    "Model Spread": games["model_spread_display"],
    "Market Spread": games["market_spread_display"],
    "ATS Pick": games["ats_pick_display"],
    "ATS Diff": games["ats_difference"],
    "ATS Units": games["ats_units"].map(lambda x: f"{x:.1f}"),
    "Model Total": games["projected_total"],
    "Market Total": games["total_line"],
    "O/U Pick": games["total_pick"],
    "Total Diff": games["total_difference"],
    "Total Units": games["total_units"].map(lambda x: f"{x:.1f}"),
    "Preseason Margin": games["preseason_expected_home_margin"],
    "Weekly Margin": games["expected_home_margin"],
    "Margin Change": games["margin_change_from_preseason"]
})

display(
    projection_board.style.format({
        "Win Prob": "{:.3f}",
        "ATS Diff": "{:.2f}",
        "Model Total": "{:.1f}",
        "Market Total": "{:.1f}",
        "Total Diff": "{:.1f}",
        "Preseason Margin": "{:.2f}",
        "Weekly Margin": "{:.2f}",
        "Margin Change": "{:+.2f}"
    }).hide(axis="index")
)


Matchup,Projected Score,SU Pick,Win Prob,Model Spread,Market Spread,ATS Pick,ATS Diff,ATS Units,Model Total,Market Total,O/U Pick,Total Diff,Total Units,Preseason Margin,Weekly Margin,Margin Change
TB @ DAL,TB 23.0 - DAL 26.3,DAL,0.590,DAL -3.2,DAL -9.5,TB +9.5,6.28,2.0,49.3,47.5,OVER,1.8,0.0,0.78,3.22,+2.44
BAL @ ATL,BAL 24.6 - ATL 23.6,BAL,0.529,BAL -1.0,ATL -2.5,BAL +2.5,3.52,1.5,48.1,44.5,OVER,3.6,1.0,-2.82,-1.02,+1.81
CHI @ GB,CHI 23.1 - GB 22.6,CHI,0.516,CHI -0.6,CHI -3.0,GB +3.0,2.43,1.0,45.7,45.5,OVER,0.2,0.0,4.35,-0.57,-4.92
CIN @ MIA,CIN 24.4 - MIA 20.4,CIN,0.612,CIN -4.0,CIN -7.0,MIA +7.0,2.97,1.0,44.8,42.5,OVER,2.3,0.5,0.06,-4.03,-4.09
CLE @ NYJ,CLE 21.6 - NYJ 21.4,CLE,0.505,CLE -0.2,NYJ -2.5,CLE +2.5,2.69,1.0,43.0,39.5,OVER,3.5,1.0,-0.56,-0.19,+0.37
DEN @ LAC,DEN 23.1 - LAC 20.6,DEN,0.570,DEN -2.5,DEN -3.5,LAC +3.5,1.01,0.5,43.7,42.5,OVER,1.2,0.0,-0.90,-2.49,-1.60
DET @ ARI,DET 27.4 - ARI 24.4,DET,0.584,DET -3.0,DET -4.5,ARI +4.5,1.49,0.5,51.8,54.5,UNDER,2.7,0.5,-5.93,-3.01,+2.92
HOU @ TEN,HOU 23.5 - TEN 19.1,HOU,0.624,HOU -4.5,HOU -7.0,TEN +7.0,2.51,1.0,42.6,39.5,OVER,3.1,0.5,-7.79,-4.49,+3.31
IND @ PIT,IND 22.9 - PIT 23.6,PIT,0.522,PIT -0.8,PIT -2.5,IND +2.5,1.73,0.5,46.5,44.5,OVER,2.0,0.0,2.32,0.77,-1.55
LV @ NE,LV 19.1 - NE 24.6,NE,0.650,NE -5.5,NE -3.5,NE -3.5,1.98,0.5,43.7,45.5,UNDER,1.8,0.0,9.37,5.48,-3.89


# Biggest Model vs Market Differences


In [30]:
biggest_ats_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Spread",
            "Market Spread",
            "ATS Pick",
            "ATS Diff"
        ]
    ]
    .sort_values("ATS Diff", ascending=False)
    .head(5)
    .copy()
)

biggest_total_differences = (
    projection_board[
        [
            "Matchup",
            "Projected Score",
            "Model Total",
            "Market Total",
            "O/U Pick",
            "Total Diff"
        ]
    ]
    .sort_values("Total Diff", ascending=False)
    .head(5)
    .copy()
)

print("BIGGEST ATS DIFFERENCES")
display(biggest_ats_differences.round(2))

print()
print("BIGGEST TOTAL DIFFERENCES")
display(biggest_total_differences.round(2))


BIGGEST ATS DIFFERENCES


,Matchup,Projected Score,Model Spread,Market Spread,ATS Pick,ATS Diff
0,TB @ DAL,TB 23.0 - DAL 26.3,DAL -3.2,DAL -9.5,TB +9.5,6.28
1,BAL @ ATL,BAL 24.6 - ATL 23.6,BAL -1.0,ATL -2.5,BAL +2.5,3.52
3,CIN @ MIA,CIN 24.4 - MIA 20.4,CIN -4.0,CIN -7.0,MIA +7.0,2.97
4,CLE @ NYJ,CLE 21.6 - NYJ 21.4,CLE -0.2,NYJ -2.5,CLE +2.5,2.69
7,HOU @ TEN,HOU 23.5 - TEN 19.1,HOU -4.5,HOU -7.0,TEN +7.0,2.51



BIGGEST TOTAL DIFFERENCES


,Matchup,Projected Score,Model Total,Market Total,O/U Pick,Total Diff
14,BUF @ LA,BUF 24.2 - LA 24.8,48.98,54.5,UNDER,5.52
1,BAL @ ATL,BAL 24.6 - ATL 23.6,48.14,44.5,OVER,3.64
4,CLE @ NYJ,CLE 21.6 - NYJ 21.4,43.02,39.5,OVER,3.52
7,HOU @ TEN,HOU 23.5 - TEN 19.1,42.61,39.5,OVER,3.11
6,DET @ ARI,DET 27.4 - ARI 24.4,51.82,54.5,UNDER,2.68


In [31]:
biggest_ats_differences["ATS Diff"] = (
    biggest_ats_differences["ATS Diff"].round(2)
)

biggest_total_differences["Model Total"] = (
    biggest_total_differences["Model Total"].round(1)
)

biggest_total_differences["Market Total"] = (
    biggest_total_differences["Market Total"].round(1)
)

biggest_total_differences["Total Diff"] = (
    biggest_total_differences["Total Diff"].round(1)
)

# Biggest Changes from the Preseason Projection

This is one of the most important weekly diagnostics. It shows which games changed most because of new in season information.


In [32]:
biggest_projection_changes = (
    projection_board[
        [
            "Matchup",
            "Preseason Margin",
            "Weekly Margin",
            "Margin Change",
            "SU Pick",
            "Win Prob"
        ]
    ]
    .assign(
        Absolute_Change=lambda x: x["Margin Change"].abs()
    )
    .sort_values("Absolute_Change", ascending=False)
    .drop(columns="Absolute_Change")
)

display(biggest_projection_changes.round(3))


,Matchup,Preseason Margin,Weekly Margin,Margin Change,SU Pick,Win Prob
12,PHI @ JAX,1.025,6.025,5.000,JAX,0.664
2,CHI @ GB,4.352,-0.569,-4.921,CHI,0.516
3,CIN @ MIA,0.060,-4.031,-4.091,CIN,0.612
9,LV @ NE,9.367,5.477,-3.890,NE,0.650
7,HOU @ TEN,-7.794,-4.489,3.305,HOU,0.624
10,MIN @ NO,-0.449,-3.576,-3.127,MIN,0.599
6,DET @ ARI,-5.933,-3.009,2.924,DET,0.584
0,TB @ DAL,0.776,3.218,2.442,DAL,0.590
13,SF @ SEA,3.760,1.821,-1.939,SEA,0.551
1,BAL @ ATL,-2.823,-1.016,1.807,BAL,0.529


In [33]:
biggest_projection_changes["Preseason Margin"] = (
    biggest_projection_changes["Preseason Margin"].round(2)
)

biggest_projection_changes["Weekly Margin"] = (
    biggest_projection_changes["Weekly Margin"].round(2)
)

biggest_projection_changes["Margin Change"] = (
    biggest_projection_changes["Margin Change"].round(2)
)

biggest_projection_changes["Win Prob"] = (
    biggest_projection_changes["Win Prob"].round(3)
)

# Sanity Checks


In [34]:
assert len(games) == len(target_schedule)
assert games["weekly_neutral_strength_diff"].notna().all()
assert games["expected_home_margin"].notna().all()
assert games["predicted_win_probability"].between(0.5, 1.0).all()
assert games["projected_total"].notna().all()

score_margin_check = (
    games["projected_home_score"]
    - games["projected_away_score"]
)

score_total_check = (
    games["projected_home_score"]
    + games["projected_away_score"]
)

assert np.allclose(
    score_margin_check,
    games["expected_home_margin"]
)

assert np.allclose(
    score_total_check,
    games["projected_total"]
)

print("All weekly game prediction sanity checks passed.")


All weekly game prediction sanity checks passed.


# Save Week Outputs

Everything is written only to the weekly layer.

The detailed parquet preserves all modeling fields for later evaluation. The CSV files provide the clean weekly projection board and largest model versus market differences.


In [35]:
detailed_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_projections.parquet"
)

current_games = games.copy()

if detailed_path.exists():

    previous_games = pd.read_parquet(
        detailed_path
    )

    current_game_ids = set(
        current_games["game_id"]
    )

    frozen_games = previous_games[
        ~previous_games["game_id"].isin(
            current_game_ids
        )
    ].copy()

    detailed_save = pd.concat(
        [
            frozen_games,
            current_games
        ],
        ignore_index=True
    )

else:

    detailed_save = current_games.copy()


detailed_save.to_parquet(
    detailed_path,
    index=False
)

projection_save = projection_board.copy()

projection_save["ATS Units"] = games["ats_units"].values
projection_save["Total Units"] = games["total_units"].values

projection_save["Win Prob"] = (
    projection_save["Win Prob"].round(3)
)

projection_save["ATS Diff"] = (
    projection_save["ATS Diff"].round(2)
)

projection_save["ATS Units"] = (
    projection_save["ATS Units"].round(1)
)

projection_save["Model Total"] = (
    projection_save["Model Total"].round(1)
)

projection_save["Market Total"] = (
    projection_save["Market Total"].round(1)
)

projection_save["Total Diff"] = (
    projection_save["Total Diff"].round(1)
)

projection_save["Total Units"] = (
    projection_save["Total Units"].round(1)
)

projection_save["Preseason Margin"] = (
    projection_save["Preseason Margin"].round(2)
)

projection_save["Weekly Margin"] = (
    projection_save["Weekly Margin"].round(2)
)

projection_save["Margin Change"] = (
    projection_save["Margin Change"].round(2)
)

projection_csv_path = (
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_projections.csv"
)

if projection_csv_path.exists():

    previous_projections = pd.read_csv(
        projection_csv_path
    )

    current_matchups = set(
        projection_save["Matchup"]
    )

    frozen_projections = previous_projections[
        ~previous_projections["Matchup"].isin(
            current_matchups
        )
    ].copy()

    projection_save = pd.concat(
        [
            frozen_projections,
            projection_save
        ],
        ignore_index=True
    )

projection_save.to_csv(
    projection_csv_path,
    index=False
)

biggest_ats_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv",
    index=False
)

biggest_total_differences.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv",
    index=False
)

biggest_projection_changes.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv",
    index=False
)


print("Saved:")
print(detailed_path)
print(projection_csv_path)

print(
    f"\nTotal saved Week {TARGET_WEEK} projections: "
    f"{len(projection_save)}"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_ats_differences.csv"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_total_differences.csv"
)

print(
    WEEK_OUTPUT_DIR
    / f"week_{TARGET_WEEK:02d}_biggest_projection_changes.csv"
)

Saved:
..\..\data\processed\weekly\week_05_projections.parquet
..\..\weekly_projections\outputs\week_05\week_05_projections.csv

Total saved Week 5 projections: 15
..\..\weekly_projections\outputs\week_05\week_05_biggest_ats_differences.csv
..\..\weekly_projections\outputs\week_05\week_05_biggest_total_differences.csv
..\..\weekly_projections\outputs\week_05\week_05_biggest_projection_changes.csv


# Weekly Pipeline

For each future week, the intended order is:

`02_Weekly_Data_Update`  
→ `03_Inseason_Feature_Engineering`  
→ `04_Weekly_Team_Strength`  
→ `06_Week2_Projections`

The original preseason notebooks remain frozen throughout the season.

The next notebook will be `06_Weekly_Evaluation.ipynb`, which will compare locked weekly predictions against actual results after each week is complete.
